# Detección de Intrusión en Tráfico MQTT/IoT

Clasificación de ataques (DoS, MitM, Intrusion) sobre el dataset **MQTTset**, usando:

- **XGBoost** (clasificación multiclase, GPU/CUDA)
- **LSTM Autoencoder** (detección de anomalías no supervisada, GPU)
- **Híbrido XGBoost + LSTM** (error de reconstrucción como feature adicional)

Dos escenarios:

1. **Caso 1 — Dataset completo**: todas las features disponibles.
2. **Caso 2 — Flujo real / cifrado (MQTTS)**: solo cabeceras de red/transporte visibles (la capa de aplicación MQTT no se ve).



## 0. Configuración y Setup


In [ ]:
import os
import math
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, roc_auc_score)

from xgboost import XGBClassifier

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ── Constantes globales ──
SEED = 42
SEQ_LEN = 10          # longitud de la ventana temporal (LSTM)
N_SPLITS = 5          # folds de validación cruzada
TEST_SIZE = 0.20      # proporción hold-out
NAN_FILL = -1.0       # valor de relleno para nulos numéricos

# ── Semillas para reproducibilidad ──
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Dispositivos ──
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
XGB_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Torch  device : {DEVICE}")
print(f"XGBoost device : {XGB_DEVICE}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    print("⚠️  ADVERTENCIA: no se detectó GPU. Activa el acelerador GPU en Settings > Accelerator.")


## 1. Carga de Datos


In [ ]:
FILES = {
    'DoS':       '/kaggle/input/datasets/alejandropenagos79/dataset/DoS.csv',
    'MitM':      '/kaggle/input/datasets/alejandropenagos79/dataset/MitM.csv',
    'Intrusion': '/kaggle/input/datasets/alejandropenagos79/dataset/Intrusion.csv',
}
TARGET_COL = 'type'

def load_and_merge(files):
    frames = []
    for attack_type, path in files.items():
        print(f"Cargando {attack_type} ...")
        df_tmp = pd.read_csv(path, low_memory=False)
        # Si el CSV no trae etiqueta, asignamos el tipo de ataque del archivo
        if not any(c in df_tmp.columns for c in ('Type', 'type', TARGET_COL)):
            df_tmp[TARGET_COL] = attack_type
        frames.append(df_tmp)
    df_merged = pd.concat(frames, ignore_index=True)
    if 'Type' in df_merged.columns:
        df_merged.rename(columns={'Type': TARGET_COL}, inplace=True)
    return df_merged

df_raw = load_and_merge(FILES)

print(f"\nDataset unificado: {df_raw.shape}")
print("\nDistribución inicial de clases:")
print(df_raw[TARGET_COL].value_counts())


## 2. Feature Engineering


In [ ]:
def calcular_entropia(texto):
    """Entropía de Shannon de una cadena (para detectar payloads aleatorios/cifrados)."""
    if pd.isna(texto) or not str(texto):
        return 0.0
    cadena = str(texto)
    probs = [c / len(cadena) for c in Counter(cadena).values()]
    return -sum(p * math.log2(p) for p in probs)

def feature_engineering(df):
    df = df.copy()

    # A. Descartar columnas de ruido / memorización de topología
    cols_ruido = [
        'ip.src', 'ip.dst', 'frame.comment', 'frame.comment.expert',
        'frame.coloring_rule.name', 'frame.coloring_rule.string',
        'frame.interface_name', 'frame.interface_id', 'frame.file_off',
    ]
    df.drop(columns=[c for c in cols_ruido if c in df.columns], errors='ignore', inplace=True)

    # B. Entropía de payload y topic
    if 'mqtt.msg' in df.columns:
        df['fe_msg_entropy'] = df['mqtt.msg'].astype(str).apply(calcular_entropia)
    if 'mqtt.topic' in df.columns:
        df['fe_topic_entropy'] = df['mqtt.topic'].astype(str).apply(calcular_entropia)
        df['fe_topic_depth'] = df['mqtt.topic'].astype(str).apply(
            lambda x: 0 if x == 'nan' else x.count('/') + 1)

    # C. Ratio de carga útil
    if 'mqtt.len' in df.columns and 'frame.len' in df.columns:
        df['mqtt.len'] = pd.to_numeric(df['mqtt.len'], errors='coerce').fillna(0)
        df['frame.len'] = pd.to_numeric(df['frame.len'], errors='coerce').fillna(0)
        df['fe_payload_ratio'] = df['mqtt.len'] / (df['frame.len'] + 1e-5)

    # D. Bandera de conexión anónima (CONNECT sin username)
    if 'mqtt.msgtype' in df.columns and 'mqtt.conflag.uname' in df.columns:
        df['mqtt.msgtype'] = pd.to_numeric(df['mqtt.msgtype'], errors='coerce').fillna(-1)
        df['mqtt.conflag.uname'] = pd.to_numeric(df['mqtt.conflag.uname'], errors='coerce').fillna(-1)
        df['fe_anon_connect'] = ((df['mqtt.msgtype'] == 1) & (df['mqtt.conflag.uname'] == 0)).astype(int)

    return df

df = feature_engineering(df_raw)
print(f"¡Feature engineering completo! Forma: {df.shape}")


## 3. Preprocesamiento


In [ ]:
def preprocess(df, target_col=TARGET_COL):
    df_prep = df.copy()

    num_cols = df_prep.select_dtypes(include=['int64', 'float64', 'int32', 'float32']).columns.tolist()
    obj_cols = df_prep.select_dtypes(include=['object', 'bool']).columns.tolist()
    for c in (target_col,):
        if c in num_cols:
            num_cols.remove(c)
        if c in obj_cols:
            obj_cols.remove(c)

    # A. Nulos numéricos -> NAN_FILL (contexto: métricas inexistentes / tramas no-MQTT)
    for col in num_cols:
        df_prep[col] = df_prep[col].fillna(NAN_FILL)

    # B. Nulos categóricos -> categoría explícita
    for col in obj_cols:
        df_prep[col] = df_prep[col].astype(str).replace({'nan': 'NOT_MQTT', 'None': 'NOT_MQTT'})
        df_prep[col] = df_prep[col].fillna('NOT_MQTT')

    # C. Categóricas a dtype 'category' (XGBoost las maneja de forma nativa)
    for col in obj_cols:
        df_prep[col] = df_prep[col].astype('category')

    # D. Codificar el target
    le_target = LabelEncoder()
    df_prep['target_encoded'] = le_target.fit_transform(df_prep[target_col])

    X = df_prep.drop(columns=[target_col, 'target_encoded'])
    y = df_prep['target_encoded']
    return X, y, le_target

X, y, le_target = preprocess(df)

print("--- Mapeo de clases del target ---")
for i, c in enumerate(le_target.classes_):
    print(f"Clase {i}: {c}")
print(f"\nX: {X.shape} | y: {y.shape} | nulos restantes: {X.isna().sum().sum()}")


### Helpers de modelado y evaluación


In [ ]:
CLASS_NAMES = le_target.classes_

def get_xgb_model(n_classes):
    """Configuración única de XGBoost (GPU si está disponible)."""
    return XGBClassifier(
        objective='multi:softprob',
        num_class=n_classes,
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        enable_categorical=True,
        tree_method='hist',
        device=XGB_DEVICE,
        random_state=SEED,
        verbosity=0,
    )

def reporte(y_true, y_pred, target_names):
    print(classification_report(y_true, y_pred, target_names=target_names, digits=4))

def plot_cm(y_true, y_pred, target_names, title, cmap='Blues'):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    plt.figure(figsize=(9, 7))
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap=cmap,
                xticklabels=target_names, yticklabels=target_names)
    plt.title(title, fontsize=14)
    plt.xlabel('Clase predicha')
    plt.ylabel('Clase real')
    plt.show()


# Caso 1 — Dataset completo (todas las features)

## 4. XGBoost con validación cruzada estratificada


In [ ]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
oof_preds = np.zeros(len(X))
oof_probs = np.zeros((len(X), len(CLASS_NAMES)))

print(f"--- XGBoost CV ({N_SPLITS} folds) en {XGB_DEVICE.upper()} ---")

for fold, (tr, val) in enumerate(skf.split(X, y)):
    model = get_xgb_model(len(CLASS_NAMES))
    model.fit(X.iloc[tr], y.iloc[tr],
              eval_set=[(X.iloc[val], y.iloc[val])], verbose=False)
    probs = model.predict_proba(X.iloc[val])
    oof_preds[val] = np.argmax(probs, axis=1)
    oof_probs[val] = probs
    print(f"Fold {fold + 1}/{N_SPLITS} - Accuracy: {accuracy_score(y.iloc[val], oof_preds[val]):.4f}")

print("\n================ REPORTE (CASO 1 - XGBoost CV) ================")
reporte(y, oof_preds, CLASS_NAMES)
plot_cm(y, oof_preds, CLASS_NAMES, 'Matriz de Confusión Normalizada - Caso 1 (XGBoost)')


## 5. Importancia de características


In [ ]:
# Se usa el último modelo entrenado en CV
df_imp = (pd.DataFrame({'Feature': X.columns, 'Importance': model.feature_importances_})
          .sort_values('Importance', ascending=False))

plt.figure(figsize=(12, 8))
sns.barplot(data=df_imp.head(20), x='Importance', y='Feature', palette='magma')
plt.title('Top 20 características - XGBoost (Caso 1)', fontsize=14)
plt.tight_layout()
plt.show()

print("\nVariables sintéticas (fe_*):")
print(df_imp[df_imp['Feature'].str.startswith('fe_')])


## 6. Diagnóstico de overfitting


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE,
                                                    random_state=SEED, stratify=y)
model = get_xgb_model(len(CLASS_NAMES))
model.fit(X_tr, y_tr)

train_acc = accuracy_score(y_tr, model.predict(X_tr))
test_acc = accuracy_score(y_te, model.predict(X_te))

print(f"Accuracy Train: {train_acc:.4%}")
print(f"Accuracy Test : {test_acc:.4%}")
print(f"Brecha        : {(train_acc - test_acc):.4%}")
print("⚠️  Overfit significativo (> 5%)" if (train_acc - test_acc) > 0.05
      else "✅  Sin sobreajuste severo")


# Caso 2 — Flujo real / cifrado (MQTTS)

Solo se conservan **cabeceras de red/transporte y tiempo**. Las métricas de la capa de aplicación MQTT (topic, msg, flags...) **no son visibles** (tráfico cifrado).


In [ ]:
COLS_RED = [
    'frame.time_delta', 'frame.time_delta_displayed', 'frame.time_relative',
    'frame.len', 'frame.cap_len', 'tcp.srcport', 'tcp.dstport',
]

def build_encrypted_features(X_full, cols):
    base = X_full[[c for c in cols if c in X_full.columns]].copy()
    for c in base.columns:
        base[c] = pd.to_numeric(base[c], errors='coerce').fillna(NAN_FILL)

    # A. Velocidad de carga (bytes/segundo)
    base['fe_bytes_per_sec'] = base['frame.len'] / (base['frame.time_delta'] + 1e-6)
    # B. Ratio captura vs longitud de cable
    base['fe_cap_ratio'] = base['frame.cap_len'] / (base['frame.len'] + 1e-6)
    # C. Puerto estándar MQTT/MQTTS
    if 'tcp.dstport' in base.columns:
        base['fe_is_standard_mqtt_port'] = base['tcp.dstport'].isin([1883, 8883]).astype(int)
    return base

X_enc = build_encrypted_features(X, COLS_RED)
print(f"Variables del escenario cifrado ({X_enc.shape[1]}): {list(X_enc.columns)}")


## 7. XGBoost sobre tráfico cifrado


In [ ]:
X_enc_tr, X_enc_te, y_enc_tr, y_enc_te = train_test_split(
    X_enc, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)

model_tls = get_xgb_model(len(CLASS_NAMES))
model_tls.fit(X_enc_tr, y_enc_tr)
y_enc_pred = model_tls.predict(X_enc_te)

print("\n================ REPORTE (CASO 2 - XGBoost TLS) ================")
reporte(y_enc_te, y_enc_pred, CLASS_NAMES)
plot_cm(y_enc_te, y_enc_pred, CLASS_NAMES,
        'Matriz de Confusión - Tráfico cifrado TLS (MQTTS)', cmap='Reds')


## 8. LSTM Autoencoder (detección de anomalías)


In [ ]:
# Solo features de red/transporte (sin feature engineering de payload)
X_tls = X[[c for c in COLS_RED if c in X.columns]].copy()
for c in X_tls.columns:
    X_tls[c] = pd.to_numeric(X_tls[c], errors='coerce').fillna(NAN_FILL)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_tls)

normal_idx = le_target.transform(['normal'])[0]

def crear_secuencias(data, labels, seq_length=SEQ_LEN):
    """Ventanas deslizantes. Etiqueta binaria: 1 si hay al menos una trama de ataque en la ventana."""
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        xs.append(data[i:i + seq_length])
        ys.append(1 if np.any(labels[i:i + seq_length] != normal_idx) else 0)
    return np.array(xs), np.array(ys)

X_seq, y_seq = crear_secuencias(X_scaled, y.values)

# Entrenar el autoencoder SOLO con tráfico normal
X_train_normal = X_seq[y_seq == 0]

train_ds = TensorDataset(torch.tensor(X_train_normal, dtype=torch.float32))
test_ds = TensorDataset(torch.tensor(X_seq, dtype=torch.float32),
                        torch.tensor(y_seq, dtype=torch.long))
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True,
                          num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=1024, shuffle=False,
                         num_workers=2, pin_memory=True)

class LSTMAutoencoder(nn.Module):
    def __init__(self, seq_len, num_features, hidden_dim=32):
        super().__init__()
        self.seq_len = seq_len
        self.encoder_lstm = nn.LSTM(num_features, hidden_dim, batch_first=True)
        self.decoder_lstm = nn.LSTM(hidden_dim, num_features, batch_first=True)

    def forward(self, x):
        _, (hidden, _) = self.encoder_lstm(x)
        latent = hidden.permute(1, 0, 2).repeat(1, self.seq_len, 1)
        output, _ = self.decoder_lstm(latent)
        return output

model_ae = LSTMAutoencoder(SEQ_LEN, X_tls.shape[1], hidden_dim=32).to(DEVICE)
criterion = nn.MSELoss(reduction='none')
optimizer = torch.optim.Adam(model_ae.parameters(), lr=0.001)

EPOCHS = 10
model_ae.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for (inputs,) in train_loader:
        inputs = inputs.to(DEVICE)
        optimizer.zero_grad()
        outputs = model_ae(inputs)
        loss = criterion(outputs, inputs).mean()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}/{EPOCHS} - Loss: {total_loss / len(train_loader):.6f}")

# Evaluación y detección de anomalías
model_ae.eval()
recon_errors, true_labels = [], []
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(DEVICE)
        outputs = model_ae(batch_x)
        mse = torch.mean((outputs - batch_x) ** 2, dim=[1, 2]).cpu().numpy()
        recon_errors.extend(mse)
        true_labels.extend(batch_y.numpy())

recon_errors = np.array(recon_errors)
true_labels = np.array(true_labels)

threshold = np.percentile(recon_errors[true_labels == 0], 95)
predicted = (recon_errors > threshold).astype(int)

print(f"Umbral de anomalía (percentil 95): {threshold:.6f}")
print("\n================ REPORTE (CASO 2 - LSTM AE) ================")
print(classification_report(true_labels, predicted,
                            target_names=['Normal', 'Ataque (Anomalía)'], digits=4))
print(f"ROC-AUC: {roc_auc_score(true_labels, recon_errors):.4f}")

plt.figure(figsize=(10, 5))
sns.histplot(recon_errors[true_labels == 0], color='blue', label='Normal',
             kde=True, stat='density', bins=50)
sns.histplot(recon_errors[true_labels == 1], color='red', label='Ataque',
             kde=True, stat='density', bins=50)
plt.axvline(threshold, color='black', linestyle='--', label=f'Umbral ({threshold:.3f})')
plt.title('Error de reconstrucción - LSTM Autoencoder', fontsize=14)
plt.legend()
plt.show()


## 9. Modelo híbrido (XGBoost + LSTM)


In [ ]:
print("--- PASO 1: Error de reconstrucción (MSE) sobre todo el dataset ---")
model_ae.eval()
full_ds = TensorDataset(torch.tensor(X_seq, dtype=torch.float32))
full_loader = DataLoader(full_ds, batch_size=1024, shuffle=False,
                         num_workers=2, pin_memory=True)

all_lstm_mse = []
with torch.no_grad():
    for (batch_x,) in full_loader:
        batch_x = batch_x.to(DEVICE)
        outputs = model_ae(batch_x)
        all_lstm_mse.extend(torch.mean((outputs - batch_x) ** 2, dim=[1, 2]).cpu().numpy())
all_lstm_mse = np.array(all_lstm_mse)

torch.cuda.empty_cache()

print("--- PASO 2: Construcción de la matriz híbrida ---")
# all_lstm_mse[i] corresponde a la ventana que TERMINA en la fila i + SEQ_LEN - 1
start = SEQ_LEN - 1
end = start + len(all_lstm_mse)
X_hybrid = X_tls.iloc[start:end].copy().reset_index(drop=True)
y_hybrid = y.iloc[start:end].copy().reset_index(drop=True)
X_hybrid['fe_lstm_mse'] = all_lstm_mse
print(f"X_hybrid: {X_hybrid.shape} | columnas: {list(X_hybrid.columns)}")

X_train_hy, X_test_hy, y_train_hy, y_test_hy = train_test_split(
    X_hybrid, y_hybrid, test_size=TEST_SIZE, random_state=SEED, stratify=y_hybrid)

# y_hybrid ya viene codificada numéricamente (target_encoded): NO se re-aplica LabelEncoder
print("--- PASO 3: XGBoost híbrido en GPU ---")
model_hybrid = get_xgb_model(len(CLASS_NAMES))
model_hybrid.fit(X_train_hy, y_train_hy)
y_pred_hy = model_hybrid.predict(X_test_hy)

print("\n================ REPORTE (CASO 2 - HÍBRIDO) ================")
reporte(y_test_hy, y_pred_hy, CLASS_NAMES)

# Importancia de features (destacando fe_lstm_mse)
importances = model_hybrid.feature_importances_
order = np.argsort(importances)[::-1]
colors = ['red' if f == 'fe_lstm_mse' else 'skyblue' for f in X_hybrid.columns[order]]

plt.figure(figsize=(10, 6))
plt.bar(range(len(importances)), importances[order], color=colors)
plt.xticks(range(len(importances)), X_hybrid.columns[order], rotation=45, ha='right')
plt.title('Importancia de features - Modelo híbrido (XGBoost + LSTM)', fontsize=14)
plt.ylabel('Importancia (Gain)')
plt.tight_layout()
plt.show()


# Resumen de resultados

> Nota: las métricas provienen de particiones distintas (OOF para el Caso 1, hold-out para el Caso 2 y predicción binaria para el autoencoder), por lo que son **orientativas** y no una comparación directa 1:1.


In [ ]:
resumen = pd.DataFrame([
    {'Modelo': 'XGBoost (completo)',      'Escenario': 'Caso 1',
     'Accuracy': accuracy_score(y, oof_preds)},
    {'Modelo': 'XGBoost (TLS)',           'Escenario': 'Caso 2',
     'Accuracy': accuracy_score(y_enc_te, y_enc_pred)},
    {'Modelo': 'LSTM Autoencoder',        'Escenario': 'Caso 2 (anomalía binaria)',
     'Accuracy': accuracy_score(true_labels, predicted),
     'ROC-AUC': roc_auc_score(true_labels, recon_errors)},
    {'Modelo': 'Híbrido XGBoost + LSTM',  'Escenario': 'Caso 2',
     'Accuracy': accuracy_score(y_test_hy, y_pred_hy)},
])

resumen['Accuracy'] = resumen['Accuracy'].map(lambda v: f'{v:.4%}')
if 'ROC-AUC' in resumen:
    resumen['ROC-AUC'] = resumen['ROC-AUC'].map(lambda v: f'{v:.4f}' if pd.notna(v) else '')

resumen


## 10. Exportación de modelos para producción (binarios)

Los modelos se guardan en binario y se empaquetan en `modelos_agente.zip` junto con todo lo necesario para que un **agente Python** los cargue sin reentrenar:

| Archivo | Contenido |
|---|---|
| `xgb_case1.ubj` | XGBoost (Caso 1, todas las features) |
| `xgb_tls.ubj` | XGBoost (Caso 2, tráfico cifrado) |
| `xgb_hybrid.ubj` | XGBoost híbrido (features + `fe_lstm_mse`) |
| `lstm_ae.pt` | LSTM Autoencoder en TorchScript (autocontenido, carga sin la clase) |
| `scaler.joblib` | `StandardScaler` de la LSTM |
| `le_target.joblib` | `LabelEncoder` del target |
| `pipeline_config.json` | orden de columnas, `SEQ_LEN`, `threshold`, clases |

> Para cargarlos desde el agente, consulta `agent_inference.py` (incluido en el zip).
> ⚠️ El agente debe reproducir **exactamente** el preprocesado (feature engineering, nulos a -1, dtype `category`, orden de columnas). Ese paso ya está resuelto en `agent_inference.py`.


In [ ]:
import os
import json
import zipfile

import joblib

OUT_DIR = '/kaggle/working/modelos_agente'
os.makedirs(OUT_DIR, exist_ok=True)

print("--- Reentrenando modelos finales sobre el 100% de los datos ---")

# 1. XGBoost Caso 1 (todas las features)
xgb_case1 = get_xgb_model(len(CLASS_NAMES))
xgb_case1.fit(X, y)

# 2. XGBoost Caso 2 (tráfico cifrado)
xgb_tls = get_xgb_model(len(CLASS_NAMES))
xgb_tls.fit(X_enc, y)

# 3. XGBoost híbrido (features + fe_lstm_mse)
xgb_hybrid = get_xgb_model(len(CLASS_NAMES))
xgb_hybrid.fit(X_hybrid, y_hybrid)

print("--- Guardando modelos XGBoost (UBJSON binario) ---")
xgb_case1.save_model(os.path.join(OUT_DIR, 'xgb_case1.ubj'))
xgb_tls.save_model(os.path.join(OUT_DIR, 'xgb_tls.ubj'))
xgb_hybrid.save_model(os.path.join(OUT_DIR, 'xgb_hybrid.ubj'))

print("--- Guardando LSTM Autoencoder (TorchScript, portable CPU/GPU) ---")
# torch.jit.script (no trace) genera un grafo independiente del dispositivo,
# necesario para que el agente cargue el modelo en CPU sin fallar por 'cuda:0'.
model_ae.cpu().eval()
scripted_ae = torch.jit.script(model_ae)
scripted_ae.save(os.path.join(OUT_DIR, 'lstm_ae.pt'))

print("--- Guardando preprocesadores (joblib) ---")
joblib.dump(scaler, os.path.join(OUT_DIR, 'scaler.joblib'))
joblib.dump(le_target, os.path.join(OUT_DIR, 'le_target.joblib'))

print("--- Guardando configuración del pipeline ---")
config = {
    'seq_len': int(SEQ_LEN),
    'nan_fill': float(NAN_FILL),
    'threshold': float(threshold),
    'normal_idx': int(normal_idx),
    'class_names': list(CLASS_NAMES),
    'feature_columns_case1': list(X.columns),
    'feature_columns_tls': list(X_enc.columns),
    'feature_columns_hybrid': list(X_hybrid.columns),
    'feature_columns_lstm_raw': list(X_tls.columns),
}
with open(os.path.join(OUT_DIR, 'pipeline_config.json'), 'w') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("--- Empaquetando en zip ---")
zip_path = '/kaggle/working/modelos_agente.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(OUT_DIR)):
        z.write(os.path.join(OUT_DIR, fn), fn)

print(f"Listo: {zip_path}")
for fn in sorted(os.listdir(OUT_DIR)):
    print(f"  - {fn}")

